# Fireworks Serverless RL: Countdown

This Colab runs the **upstream** [`training/examples/serverless_rl/countdown_rl.py`](https://github.com/fw-ai/cookbook/blob/main/training/examples/serverless_rl/countdown_rl.py) implementation without copying or modifying its training loop. It:

- sparse-checks out only `training/` from `fw-ai/cookbook`;
- installs that checkout with `pip install --pre -e training`;
- imports the upstream `Config` and `ServerlessCountdownRL` classes;
- uses the bundled 32-row `data/countdown_train.jsonl` sample;
- explicitly leaves Router Replay enabled; and
- keeps all paid training behind `RUN_TRAINING = False`.

> **Cost warning:** setup and configuration cells are safe, but changing `RUN_TRAINING` to `True` creates a real Fireworks serverless training session, samples model completions, and performs optimizer steps that may incur charges.

## 1. Fetch only the upstream training tree

The checkout is recreated on every full run so the notebook executes the current `main` revision from GitHub. `--filter=blob:none` avoids downloading file contents that are not needed, while sparse checkout materializes only `training/`. The final command prints the exact commit used, which is useful when comparing runs.

For a reproducible experiment, replace `main` below with a reviewed commit SHA.

In [ ]:
%%bash
set -euxo pipefail

REPO_DIR=/content/fw-cookbook
rm -rf "${REPO_DIR}"
git clone --filter=blob:none --no-checkout https://github.com/fw-ai/cookbook.git "${REPO_DIR}"
git -C "${REPO_DIR}" sparse-checkout init --no-cone
git -C "${REPO_DIR}" sparse-checkout set /training/
git -C "${REPO_DIR}" checkout main

echo "Upstream commit: $(git -C "${REPO_DIR}" rev-parse HEAD)"

## 2. Install the upstream package

`--pre` allows the pre-release SDK versions required by this example. `-e training` makes imports resolve directly to the sparse checkout, so `Config` and the RL class below come from the upstream file rather than a notebook reimplementation. Colab may show dependency-resolution warnings while replacing preinstalled packages.

In [ ]:
%cd /content/fw-cookbook
%pip install --pre -e training

## 3. Load the API key without putting it in the notebook

In Colab, open **Secrets** (the key icon), add a secret named `FIREWORKS_API_KEY`, and enable notebook access. This cell copies it into the environment expected by the upstream `Config`. It does not print the key. A missing key is fine until paid training is enabled. It also enables Hugging Face remote tokenizer code, which the upstream default Kimi K3 tokenizer requires.

In [ ]:
import os

# Required by the custom tokenizer used by the upstream Kimi K3 default.
os.environ["HF_TRUST_REMOTE_CODE"] = "1"

try:
    from google.colab import userdata
    fireworks_api_key = userdata.get("FIREWORKS_API_KEY")
except Exception:
    # Also supports non-Colab environments where the variable is already set.
    fireworks_api_key = os.environ.get("FIREWORKS_API_KEY", "")

if fireworks_api_key:
    os.environ["FIREWORKS_API_KEY"] = fireworks_api_key
    print("FIREWORKS_API_KEY is available.")
else:
    print("No FIREWORKS_API_KEY found; setup can continue, but training cannot run.")

## 4. Import the exact upstream implementation

The source assertion prevents an accidentally installed copy elsewhere in the Python environment from being used. The dataset assertion similarly ensures the run uses the small JSONL file bundled beside the upstream script—not the much larger Hugging Face dataset used by the upstream default.

In [ ]:
import inspect
from pathlib import Path

from training.examples.serverless_rl.countdown_rl import Config, ServerlessCountdownRL

REPO_ROOT = Path("/content/fw-cookbook").resolve()
UPSTREAM_SCRIPT = REPO_ROOT / "training/examples/serverless_rl/countdown_rl.py"
DATASET = REPO_ROOT / "training/examples/serverless_rl/data/countdown_train.jsonl"

assert Path(inspect.getfile(ServerlessCountdownRL)).resolve() == UPSTREAM_SCRIPT
assert DATASET.is_file(), f"Bundled dataset not found: {DATASET}"

row_count = sum(1 for line in DATASET.open() if line.strip())
print(f"Implementation: {UPSTREAM_SCRIPT}")
print(f"Bundled dataset: {DATASET} ({row_count} rows)")

## 5. Configure the run

Only three upstream defaults are made explicit here:

1. `dataset` points to the bundled sample requested for this notebook.
2. `router_replay=True` preserves Router Replay. For an MoE model, rollout routing matrices are requested, validated, and replayed during training; the upstream implementation automatically skips this for dense models after checking model metadata.
3. `run_dir` puts metrics and artifacts in a predictable Colab directory.

All other behavior—including Kimi K3, LoRA rank, 20 optimizer steps, 16 prompt groups per step, 8 completions per group, GRPO-style within-group advantages, held-out evaluation, checkpointing, and reward plotting—comes from the current upstream `Config` defaults. The sample has 32 rows; the default 16-row held-out split leaves 16 rows in the cycling training pool.

The displayed configuration redacts the API key. Change fields here before enabling the paid cell if you intentionally want a smaller or different run.

In [ ]:
from dataclasses import asdict
from pprint import pprint

cfg = Config(
    dataset=str(DATASET),
    router_replay=True,
    run_dir="/content/serverless_rl_run",
)

assert cfg.router_replay is True
assert Path(cfg.dataset).resolve() == DATASET

visible_config = asdict(cfg)
visible_config["api_key"] = "<set>" if cfg.api_key else "<missing>"
pprint(visible_config, sort_dicts=False)

## 6. Explicit paid-run gate

This is the only cell that constructs `ServerlessCountdownRL` or calls `run()`. Construction connects to the serverless service and creates the LoRA training client, so both operations remain inside the guard.

When enabled, the unmodified upstream loop repeatedly:

1. saves current LoRA weights for a sampler;
2. samples grouped Countdown completions (including MoE routes for Router Replay);
3. scores equations with the upstream composite reward;
4. standardizes rewards within each prompt group and drops zero-signal groups;
5. performs importance-sampling forward/backward and an Adam optimizer step; and
6. evaluates the same deterministic held-out slice at configured intervals.

Review the printed configuration and expected Fireworks cost, then deliberately change the flag to `True`.

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    if not cfg.api_key:
        raise RuntimeError("Set the FIREWORKS_API_KEY Colab secret before starting training.")
    trainer = ServerlessCountdownRL(cfg)
    trainer.run()
else:
    print("Training is disabled. Set RUN_TRAINING = True only when you intend to start a paid run.")

## 7. Inspect artifacts after a run

The upstream class writes line-delimited training metrics, fixed-evaluation metrics and completions, lifecycle/checkpoint references, and (when plotting is available) `reward_curve.png`. This cell is read-only and is safe when training has not run.

In [ ]:
from IPython.display import Image, display

run_dir = Path(cfg.run_dir)
if not run_dir.exists():
    print(f"No artifacts yet: {run_dir}")
else:
    for path in sorted(p for p in run_dir.rglob("*") if p.is_file()):
        print(path.relative_to(run_dir))

    reward_curve = run_dir / "reward_curve.png"
    if reward_curve.exists():
        display(Image(filename=str(reward_curve)))

## Reading the results

- **`eval/raw_reward` is the learning curve to trust.** It evaluates the same held-out prompts throughout the run. Training reward comes from changing batches and the optimizer subset excludes zero-variance groups.
- **`rollout/filter_ratio` shows how much data supplied a useful group-relative signal.** A group where every completion receives the same reward is dropped.
- **`train/router_replay` confirms whether Router Replay was active.** It remains requested in this notebook, but upstream intentionally disables it for dense models.
- **Sampler and trainer-state checkpoints differ.** Sampler snapshots support rollouts/promotion; DCP trainer-state checkpoints support optimizer resume. The default notebook configuration does not promote a model.

Because this notebook imports the live upstream module, behavior may change when upstream `main` changes. Record the printed commit with experiment results, and pin a SHA when reproducibility matters.